# Handed the Fix, Still Failing: reproduce the tables

Recomputes every table and figure of the paper from the released run outputs (`runs/<run_id>/output/results.jsonl`, the trace archives and the label files). No GPU and no competition data are needed. Turn Internet on.

The campaigns themselves (13 arms x 127 tasks on Kaggle L4 x4) are rebuilt with `tools/kernel.py`; see the repository README.

In [ ]:
REPO = "https://github.com/davidquicast/gemma4-oracle-ladder"  # public repository URL
assert REPO, "set REPO to the public repository URL"
!git clone --depth 1 $REPO repo
%cd repo

## 1. Solve rates and paired tests, recomputed from the raw result rows

In [ ]:
!python tools/analyze.py wave1
!python tools/analyze.py wave1 --reference base_L2 --out runs/wave1/analysis_vs_L2.json
!python tools/analyze.py wave1 --reference base_L3 --out runs/wave1/analysis_vs_L3.json
!python tools/analyze.py wave1 --complete-hints base_L3 --out runs/wave1/analysis_complete_L3.json
!python tools/analyze.py wave2 --reference base_r2_L3
!python tools/analyze.py wave1 wave2 --reference base_L3 --out runs/wave2/analysis_vs_L3.json
!python tools/analyze.py wave1 wave2 --reference base_L0 --out runs/wave2/analysis_vs_L0.json
!python tools/analyze.py wave1 wave2 --complete-hints base_L3 --reference base_L3 --out runs/wave2/analysis_complete_L3.json
# wave 3 (E2B, hunk protocol) and wave 3b (E4B): the project plan the work plan and 7
!python tools/analyze.py wave3 --reference hunk_L3
!python tools/analyze.py wave1 wave2 wave3 --reference base_L3 --out runs/wave3/analysis_vs_L3.json
!python tools/analyze.py wave1 wave3 --reference base_L2 --out runs/wave3/analysis_vs_L2.json
!python tools/analyze.py wave1 wave2 wave3 --reference base_L0 --out runs/wave3/analysis_vs_L0.json
!python tools/analyze.py wave1 wave2 wave3 --complete-hints base_L3 --reference base_L3 --out runs/wave3/analysis_complete_L3.json
!python tools/protocol_test.py runs/wave3/protocol_test.json --treat wave3:hunk_L3 wave3:hunk_r2_L3 --control wave1:base_L3 wave2:base_r2_L3
!python tools/protocol_test.py runs/wave3/protocol_test_patch_posthoc.json --pairs-only --treat wave2:patch_L3 --control wave1:base_L3
!python tools/protocol_test.py runs/wave3/carryover_test.json --key L2 --mcnemar-rule --treat wave3:hunk_L2 --control wave1:base_L2
!python tools/protocol_test.py runs/wave3/carryover_test.json --key L0 --mcnemar-rule --treat wave3:hunk_L0 --control wave1:base_L0 wave2:base_r2_L0
!python tools/analyze.py wave3b --reference base_e4b_L3
!python tools/protocol_test.py runs/wave3b/pairing.json --pairs-only --treat wave3b:hunk_e4b_L3 --control wave3b:base_e4b_L3
# wave 4 (E4B at L0 and L2): the project plan the work plan
!python tools/analyze.py wave4 --reference base_e4b_L0
!python tools/analyze.py wave3b wave4 --reference base_e4b_L2 --out runs/wave4/analysis_vs_L2.json

## 2. `git apply` outcomes of the patch arm, recounted from its traces

In [ ]:
!python tools/patch_channel.py wave2 patch_L3 | head -40
!for a in hunk_L3 hunk_r2_L3 hunk_L2 hunk_L0; do python tools/patch_channel.py wave3 $a > /dev/null && echo "wave3 $a ok"; done
!python tools/patch_channel.py wave3b hunk_e4b_L3 > /dev/null && echo "wave3b hunk_e4b_L3 ok"

## 3. The paper's tables

In [ ]:
!python tools/paper_tables.py

## 4. Figures

In [ ]:
!python tools/figures.py
from IPython.display import Image, display
display(Image('paper/figures/fig1_ladder.png'), Image('paper/figures/fig2_patch_size.png'))

## 5. Evidence ledger

Every bracketed id in the paper is a claim pinned to a file by SHA-256. After the recomputation above the files must be byte-identical, so validation still passes.

In [ ]:
!python tools/ledger.py validate
!git status --short runs | head